# M2 · Profile the bottleneck (compute vs memory)

**Question:** Should we optimize text encoding, GEMM, attention, or launch overhead? Attribute kernel classes and the warm critical path using two different records. Keep GPU-busy percentages separate from end-to-end wall time.

**Learning loop:** predict → inspect frozen evidence → compute → critique → propose a controlled experiment.

Reference lesson: [open course/M02_profiling/LESSON.md](../course/M02_profiling/LESSON.md). Switch language: [中文](./02_profile_and_bottlenecks.zh-CN.ipynb).

**Default mode:** CPU-only, internet-free, read-only. GPU work is never run implicitly.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## Inspect and recompute real evidence

Run cells in order. Explain every denominator and scope.

In [ ]:
trace = ns.read_json("evidence/results/p2-profile-profile/rocprof-dit/analysis.json")
rows = trace["by_class"]
assert trace["gpu_busy_pct"] == 97.8
ranked = sorted(rows.items(), key=lambda kv: kv[1]["total_s"], reverse=True)
for name, row in ranked:
    print(f"{name:24s} {row['share_of_busy_pct']:5.1f}% GPU-busy / {row['total_s']:.2f}s traced")
plot_values = {name: row["share_of_busy_pct"] for name, row in ranked[:4]}
plot_ylabel, plot_title = "% GPU busy", "Kernel composition (traced; not E2E)"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
pipe = ns.read_json("evidence/results/p2-profile-profile/pipeline_breakdown.json")
total = pipe["T_total_warm_s"]
dit = pipe["T_dit_denoise_s"]
share = dit / total
print(f"Warm critical-path DiT: {dit:.3f}/{total:.3f} = {share:.2%}")
print("Cached TE in warm requests:", pipe["T_text_encoder_cached_s"], "s")
assert 0.95 < share < 1.0


## Your turn / Self-check

State two distinct bottleneck percentages and their denominators. Why does 61.3% GEMM of GPU busy time NOT mean 61.3% of the whole user request?

**Evidence discipline:** Answer in your own words. This notebook's numbers come from existing committed records, not your GPU. If your optional smoke runs, its data must remain under `runs/` and may not be presented as a full-protocol median.
